# Digital options

`tq.DigitalOption` (cash-or-nothing, asset-or-nothing) priced three ways: Black–Scholes closed form, Monte Carlo
(`VanillaMCPricer`), and replication by a call spread of vanillas, where the smile skew shows up.
Docs: *Models and Pricing* (Digital options), *Exercises* (`equity.tex`).

In [ ]:
import os
from datetime import timedelta
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorquant as tq

## Market

In [ ]:
mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")   # latest snapshot
EVALUATION_DATE = tq.Settings.evaluation_date = mkt.evaluation_date
eq = mkt.get_equity("TICKER1")
EXPIRY = EVALUATION_DATE + timedelta(days=365)
T = eq.year_fraction(EXPIRY)
S0, F, DF = float(eq.spot), float(eq.forward(EXPIRY)), float(eq.discount_curve.discount(EXPIRY))
CASH, ASSET = tq.PayoffType.CashOrNothing, tq.PayoffType.AssetOrNothing
print(f"S0 = {S0:.2f}  F(1Y) = {F:.2f}  P(0,1Y) = {DF:.5f}")


def digital(option_type, K, payoff_type=CASH, cash=1.0):
    return tq.DigitalOption(eq.currency, EVALUATION_DATE, EXPIRY, option_type, K, payoff_type=payoff_type,
                            cash_amount=cash, underlying=eq.ticker)


def vanilla(option_type, K):
    return tq.VanillaOption(eq.currency, start_date=EVALUATION_DATE, end_date=EXPIRY, option_type=option_type,
                            strike=K, underlying=eq.ticker)


def bs(product, autodiff=False):
    pricer = tq.BlackScholesPricer()
    pricer.price(product, mkt, autodiff=autodiff)
    return float(product.price), pricer

## Black–Scholes
Cash-or-nothing $A\,P(0,T)\,N(\phi d_2)$ ($N(\phi d_2)$: risk-neutral probability of finishing in the money),
asset-or-nothing $P(0,T)\,F\,N(\phi d_1)$, with the vol at the strike. Parities: cash call + cash put $= A\,P(0,T)$;
asset call + asset put $= P(0,T)\,F$; asset-or-nothing $- K\,\times$ cash-or-nothing $=$ vanilla.

In [ ]:
rows = []
for m in (0.8, 0.9, 1.0, 1.1, 1.2):
    K = round(m * F, 2)
    cc, cp = bs(digital(tq.OptionType.Call, K))[0], bs(digital(tq.OptionType.Put, K))[0]
    ac = bs(digital(tq.OptionType.Call, K, ASSET))[0]
    vc = bs(vanilla(tq.OptionType.Call, K))[0]
    rows.append({"K/F": m, "cash call": cc, "cash put": cp, "call+put - DF": cc + cp - DF,
                 "asset call": ac, "asset - K cash - vanilla": ac - K * cc - vc})
pd.DataFrame(rows).set_index("K/F")

## Call-spread replication
A digital call is the limit of a call spread: $D(K) \approx \frac{C(K-h) - C(K+h)}{2h} \to -\partial C/\partial K$. On a smile
the vanilla vol depends on $K$, so
$$-\frac{\partial C}{\partial K} = \underbrace{P(0,T)\,N(d_2)}_{\text{BS digital at }\sigma(K)} \;-\; \mathcal{V}\,\frac{\partial\sigma}{\partial K}.$$
With a negative skew ($\partial\sigma/\partial K < 0$) the market digital call is worth **more** than the Black–Scholes one.

In [ ]:
K = round(F, 2)
vs = eq.vol_surface
bs_digital, _ = bs(digital(tq.OptionType.Call, K))
opt = vanilla(tq.OptionType.Call, K)
pricer = tq.BlackScholesPricer(); pricer.price(opt, mkt, autodiff=True)
vega = float(tf.reduce_sum(pricer.tape.gradient(opt.price, vs.volatility_matrix)))
slope = (float(vs.volatility(K * 1.001, T)) - float(vs.volatility(K * 0.999, T))) / (0.002 * K)
rows = []
for frac in (0.10, 0.05, 0.02, 0.01, 0.001):
    h = frac * K
    spread = (bs(vanilla(tq.OptionType.Call, K - h))[0] - bs(vanilla(tq.OptionType.Call, K + h))[0]) / (2 * h)
    rows.append({"h / K": frac, "call spread": spread})
print(f"BS digital (vol at K) = {bs_digital:.5f}   skew term -vega * dsigma/dK = {-vega * slope:+.5f}   "
      f"sum = {bs_digital - vega * slope:.5f}")
pd.DataFrame(rows).set_index("h / K").round(5)

Across strikes: the gap is largest where the smile is steepest. The call-spread digital **jumps at the strike nodes** of the
surface: the vols are linear between nodes, so $\partial\sigma/\partial K$ is piecewise constant and the digital (a derivative
of the vanilla price) inherits its jumps. A digital is priced off the *slope* of the smile: the interpolation matters.

In [ ]:
strikes = F * np.linspace(0.7, 1.3, 25)
bs_d, spread_d = [], []
for k in strikes:
    bs_d.append(bs(digital(tq.OptionType.Call, k))[0])
    h = 0.005 * k
    spread_d.append((bs(vanilla(tq.OptionType.Call, k - h))[0] - bs(vanilla(tq.OptionType.Call, k + h))[0]) / (2 * h))
plt.figure(figsize=(7, 4))
plt.plot(strikes / F, bs_d, label="Black–Scholes (vol at K)")
plt.plot(strikes / F, spread_d, label="call spread (smile)")
plt.title("Digital call, 1Y"); plt.xlabel("K / F"); plt.grid(alpha=0.3); plt.legend();

Hedging with a finite spread: $\frac{C(K-h) - C(K)}{h} \ge D(K) \ge \frac{C(K) - C(K+h)}{h}$ (super- and sub-replication: the
first pays at least 1 above $K$, the second at most 1). The tighter $h$, the closer the bounds, the larger the notional
of vanillas ($1/h$) to trade.

In [ ]:
c = lambda k: bs(vanilla(tq.OptionType.Call, k))[0]
pd.DataFrame([{"h / K": f, "sub": (c(K) - c(K + f * K)) / (f * K), "super": (c(K - f * K) - c(K)) / (f * K),
               "calls per unit of payoff": 1 / (f * K)} for f in (0.05, 0.02, 0.01)]).set_index("h / K").round(5)

## Greeks of a digital
Cash-or-nothing call, unit amount, Black–Scholes by AAD (gamma: second tape), against $S/K$ at three maturities:
$\Delta = P(0,T)\,\dfrac{n(d_2)}{S\sigma\sqrt T}$ (a bump centred on the strike, sharper as $T \to 0$);
$\Gamma = -P(0,T)\,\dfrac{n(d_2)\,d_1}{S^2\sigma^2 T}$, **positive below the strike and negative above**;
$\mathcal{V} = -P(0,T)\,n(d_2)\,\dfrac{d_1}{\sigma}$, **positive out of the money and negative in the money**: the vega flips
sign at $d_1 = 0$ (spot $\approx K e^{-(r-q+\sigma^2/2)T}$), where the vanilla has its maximum vega.

In [ ]:
K_g = round(F, 2)
spot_var, s0 = eq.spot, float(eq.spot)


def digital_greeks(expiry_days):
    out = []
    for x in np.linspace(0.6, 1.4, 81):
        spot_var.assign(x * K_g)
        d = tq.DigitalOption(eq.currency, EVALUATION_DATE, EVALUATION_DATE + timedelta(days=expiry_days),
                             tq.OptionType.Call, K_g, underlying=eq.ticker)
        pricer = tq.BlackScholesPricer()
        with tf.GradientTape() as outer:
            pricer.price(d, mkt, autodiff=True)
            delta, vega = pricer.tape.gradient(d.price, [spot_var, eq.vol_surface.volatility_matrix])
        gamma = outer.gradient(delta, spot_var)
        out.append({"S/K": x, "price": float(d.price), "delta": float(delta),
                    "gamma": float(gamma), "vega": float(tf.reduce_sum(vega)) / 100})   # vega per vol point
    spot_var.assign(s0)
    return pd.DataFrame(out).set_index("S/K")


greeks_by_t = {f"{d // 30}M" if d < 365 else "1Y": digital_greeks(d) for d in (30, 91, 365)}
fig, axes = plt.subplots(1, 4, figsize=(17, 3.6))
for ax, g in zip(axes, ("price", "delta", "gamma", "vega")):
    for label, df in greeks_by_t.items():
        ax.plot(df.index, df[g], label=label)
    ax.axvline(1.0, color="grey", lw=0.8, ls=":"); ax.axhline(0.0, color="grey", lw=0.6)
    ax.set_title(g); ax.set_xlabel("S / K"); ax.grid(alpha=0.3)
axes[0].legend(); plt.tight_layout()

In [ ]:
one_year = greeks_by_t["1Y"]
flip = one_year.index[np.argmax(np.diff(np.sign(one_year["vega"])) != 0)]
print(f"1Y: vega changes sign near S/K = {flip:.3f}; gamma changes sign near S/K = "
      f"{one_year.index[np.argmax(np.diff(np.sign(one_year['gamma'])) != 0)]:.3f}")
print(f"1M: max delta {greeks_by_t['1M']['delta'].max():.4f} per unit of payoff (vs {one_year['delta'].max():.4f} at 1Y)")

**Hedging a digital.** Delta-hedging it alone does not work near expiry: the delta is a spike of height
$\sim 1/(S\sigma\sqrt T)$ at the strike and the gamma flips sign there (pin risk), so the hedge swings from zero to a large
position on small spot moves. The standard way:

- **Replicate with a call spread**, not with the underlying: long $1/h$ calls at $K - h$ (or $K - h/2$), short $1/h$ at $K$
  (or $K + h/2$). The spread carries the digital's delta, gamma *and* vega (including the vega flip and the skew term), and
  its greeks stay bounded at expiry (gamma $\le 1/h$ per unit). Hedge the residual (spread vs digital) dynamically.
- **Choose the side and the width conservatively**: sell the digital and buy the over-replicating spread
  $[K - h, K]$ (it pays at least the digital: the difference is a reserve); $h$ is set by the liquidity of the vanillas
  and by the gamma / pin risk the book can carry ($1/h$ vanillas per unit of payoff).
- **Price and risk on the smile**: the hedge is made of vanillas at $K \pm h$, so the digital must be valued with the
  skew term $-\mathcal{V}\,\partial\sigma/\partial K$ (call spread on the surface, or a smile model), otherwise the vega hedge
  of the spread does not match.
- **Near expiry / near the strike**: keep the spread static and stop re-hedging the digital itself (the dynamic
  delta hedge of the indicator would trade huge amounts on noise).

## Monte Carlo
`VanillaMCPricer` on a `DigitalOption`. GBM with the vol at $K$ reproduces Black–Scholes. Local vol simulates the whole
smile, so it captures the skew term; it builds its own smooth smile (spline in total variance), whose slope at $K$ differs
from the piecewise-linear one of the surface: close to the call spread, not equal (about 0.009 here, the same with
more steps).

In [ ]:
def mc(model, smoothing=0.0, n_steps=1, autodiff=False, n_paths=200_000):
    d = digital(tq.OptionType.Call, K)
    pricer = tq.VanillaMCPricer(model=model, n_paths=n_paths, n_steps=n_steps, seed=7, digital_smoothing=smoothing)
    pricer.price(d, mkt, autodiff=autodiff)
    return d, pricer

gbm, _ = mc(tq.GeometricBrownianMotion)
lv, _ = mc(tq.LocalVolatilityModel, n_steps=100)
h = 0.005 * K
market = (c(K - h) - c(K + h)) / (2 * h)
print(f"GBM MC     {float(gbm.price):.5f} +/- {float(gbm.standard_error):.5f}   BS digital  {bs_digital:.5f}")
print(f"LocalVol MC {float(lv.price):.5f} +/- {float(lv.standard_error):.5f}   call spread {market:.5f}")

Greeks: with the hard indicator the pathwise derivative is zero almost everywhere, AAD gives no delta (`None`). A
sigmoid of width $\varepsilon K$ (`digital_smoothing`) makes the payoff smooth: AAD delta, small price bias.

In [ ]:
ref = digital(tq.OptionType.Call, K)
ref_pricer = tq.BlackScholesPricer()
ref_pricer.price(ref, mkt, autodiff=True)
bs_delta = float(ref_pricer.tape.gradient(ref.price, eq.spot))
rows = []
for eps in (0.0, 0.002, 0.01, 0.05):
    d, pricer = mc(tq.GeometricBrownianMotion, smoothing=eps, autodiff=True)
    g = pricer.tape.gradient(d.price, eq.spot)
    rows.append({"smoothing": eps, "price": float(d.price), "price - BS": float(d.price) - bs_digital,
                 "delta (AAD)": None if g is None else float(g), "BS delta": bs_delta})
pd.DataFrame(rows).set_index("smoothing").round(5)

## Exercises
1. Show that asset-or-nothing call $- K\times$ cash-or-nothing call $=$ vanilla call, and explain why it holds for any model.
2. Why does a negative skew make the digital call more expensive than Black–Scholes, and the digital put cheaper?
3. A trader hedges a digital call with a 1% call spread: how much does the hedge cost above the digital, and what is the
   risk at expiry if the spot ends between $K - h$ and $K$?